[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Dunder Methods


## What you will be able to do

Make your own object print readably, compare by its contents rather than by identity, report its
length, and sort. All by writing methods that Python calls for you.


## The idea

### The problem

Two things about the `Station` class have been left broken since it was written.

Print one, and you get `<__main__.Station object at 0x7f8b1c0d5e50>`. That is the class name and the
object's address in memory, and neither is anything you wanted to know. Put three stations in a
list and print the list, and you get three of those. The one question you actually have while
debugging, what is this object holding, is the one question the output does not answer.

Compare two stations that hold identical data and you get `False`. They have the same name, the
same readings and the same unit, and Python says they are different, because by default `==` on
your own class asks whether the two names refer to one object rather than whether their contents
match. The **Why Classes** notebook showed `is` doing exactly that, deliberately. Here it is the
default for `==` too, and it is rarely what you want.

Neither is a bug in Python. It has no way to guess how you would like a station printed, or which
attributes decide whether two stations count as equal. Both are questions only you can answer, and
there is a place to answer them.

### What a dunder method is

> A **dunder method** is a method whose name begins and ends with two underscores. You do not call
> it. Python calls it on your behalf when your object turns up in a particular situation: being
> printed, compared, measured, added, or used in a `for` loop. Writing `__len__` is what makes
> `len(x)` work.

### Why it works that way

Python decides what an operation does by looking for a method, not by checking a type. `len(x)`
does not test whether `x` is a list. It looks for `x.__len__` and calls it, and anything that has
one works. That single design choice is why your own class can be printed, compared, sorted and
measured with the same syntax as a built-in, and why you do not have to learn a separate vocabulary
for your own types.

The pair to understand first is `__repr__` and `__str__`, because there are two of them for a
reason.

`__repr__` is for you. It should be unambiguous, and the convention is to make it look like the
code that would recreate the object: `Station('Tromso', [-4.1], 'C')`. When you are staring at
something unexpected in a list, that tells you everything.

`__str__` is for whoever reads the program's output, and can leave things out: `Tromso (3 readings,
C)`.

If you write only one, write `__repr__`. `str()` falls back to it when `__str__` is absent, so one
method covers both. It does not work the other way, which produces a trap covered below.

Equality has a consequence worth knowing before you write it. Defining `__eq__` makes your objects
unusable in a set or as dictionary keys, until you also write `__hash__`. That is not Python being
awkward; it is protecting a rule those containers depend on, and the reason is in the errors.

### Where you will meet this

In every library, doing exactly this job. A `Path` prints as `PosixPath('data.csv')`, which is its
`__repr__`. A pandas `DataFrame` prints as a table, which is its `__repr__` running. `datetime`
objects compare and sort with `<` because they define the methods for it. When a library's object
prints nicely, somebody wrote the method you are about to write.

### What this notebook covers

`__repr__` and `__str__` and which one runs when, `__eq__` and the hashing rule that comes with it,
`__len__` and what it does to truth testing, and `__lt__` so your objects can be sorted. Then the
errors, which are mostly about writing one of a pair and not the other.

### A first look

Three lines added to a class. There is nothing to run yet: read it, and read the output underneath
it.

```python
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r})"


print([Station("Tromso", [-4.1]), Station("Bodo", [-2.6])])
```

```
[Station('Tromso', [-4.1]), Station('Bodo', [-2.6])]
```

Without `__repr__` that line prints two memory addresses.


## Setup

One import, a helper, and the class as it currently stands.

- `re` is used only by the helper below, to hide memory addresses

`masked` exists for one reason. The default way an object prints includes its address in memory,
which is different every time the notebook runs. Printing it directly would make this notebook's
saved output disagree with what you see on your own screen, so the address is replaced with `0x...`
wherever it appears. Everything else in the output is real.

`Bare` is `Station` with no dunder methods at all, which is where the notebook starts.

**Run this cell before the rest of the notebook.**


In [1]:
import re


def masked(value):
    """repr(value) with any memory address hidden, so the output is the same every run."""
    return re.sub(r"0x[0-9a-f]+", "0x...", repr(value))


class Bare:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit


print(masked(Bare("Tromso", [-4.1, -2.6, -3.8])))


<__main__.Bare object at 0x...>


## Worked examples

### What you get with nothing defined

Every way of turning the object into text gives the same unhelpful answer.


In [2]:
north = Bare("Tromso", [-4.1, -2.6, -3.8])

print("print:    ", masked(north))
print("repr:     ", masked(north))
print("in a list:", masked([north, north]))

first = Bare("Tromso", [-4.1], "C")
second = Bare("Tromso", [-4.1], "C")
print()
print("identical contents, first == second:", first == second)
print("and first is second:               ", first is second)


print:     <__main__.Bare object at 0x...>
repr:      <__main__.Bare object at 0x...>
in a list: [<__main__.Bare object at 0x...>, <__main__.Bare object at 0x...>]

identical contents, first == second: False
and first is second:                False


Two objects holding the same three values, and `==` says they differ.

That is the default `__eq__`, which compares identity. It is the correct default, because Python
cannot know whether two stations with the same name are the same station or two records of it.

### `__repr__`: make it look like the call that would rebuild it

The convention is worth following exactly. A repr that reads as a constructor call can be copied
out of a log, pasted into a cell, and run.

`!r` inside the f-string is what puts the quotes around the string and the brackets around the
list. It formats each value with its own `repr` rather than its `str`, which is what makes the
output valid Python rather than `Station(Tromso, [-4.1], C)`.


In [3]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r}, {self.unit!r})"


north = Station("Tromso", [-4.1, -2.6, -3.8])

print(repr(north))
print(north)


Station('Tromso', [-4.1, -2.6, -3.8], 'C')
Station('Tromso', [-4.1, -2.6, -3.8], 'C')


Both lines are the same, because `str` falls back to `__repr__` when there is no `__str__`.

### Where `__repr__` earns its place

Printing one object was already tolerable. The difference shows up when the object is inside
something else.


In [4]:
group = [north, Station("Bodo", [-2.6], "C")]

print("a list: ", group)
print("a dict: ", {"north": north})
print("a tuple:", (north,))


a list:  [Station('Tromso', [-4.1, -2.6, -3.8], 'C'), Station('Bodo', [-2.6], 'C')]
a dict:  {'north': Station('Tromso', [-4.1, -2.6, -3.8], 'C')}
a tuple: (Station('Tromso', [-4.1, -2.6, -3.8], 'C'),)


A container prints its contents with `repr`, never with `str`. That is deliberate: a list is a
programmer's view of data, so it shows the unambiguous form.

This is the reason `__repr__` is the one to write first. Most of the time you are looking at your
objects, they are inside a list, a dictionary, or a traceback.

### `__str__`: the version for people

Add `__str__` when the friendly form differs from the debugging form.


In [5]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r}, {self.unit!r})"

    def __str__(self):
        return f"{self.name} ({len(self.readings)} readings, {self.unit})"


north = Station("Tromso", [-4.1, -2.6, -3.8])

print("print(north): ", north)
print("str(north):   ", str(north))
print("repr(north):  ", repr(north))
print("in a list:    ", [north])
print(f"f-string:      {north}")
print(f"f-string with !r: {north!r}")


print(north):  Tromso (3 readings, C)
str(north):    Tromso (3 readings, C)
repr(north):   Station('Tromso', [-4.1, -2.6, -3.8], 'C')
in a list:     [Station('Tromso', [-4.1, -2.6, -3.8], 'C')]
f-string:      Tromso (3 readings, C)
f-string with !r: Station('Tromso', [-4.1, -2.6, -3.8], 'C')


`print` and f-strings use `__str__`. `repr()`, containers and the notebook's own display of a
returned value use `__repr__`. `!r` inside an f-string asks for the repr explicitly, which is the
same `!r` used to build the repr above.

### Which method runs when

| You write | Python calls it for |
|---|---|
| `__repr__` | `repr(x)`, and `x` inside any printed list, dict or tuple |
| `__str__` | `print(x)`, `str(x)`, `f"{x}"` |
| `__eq__` | `x == y`, and `x in collection` |
| `__hash__` | `{x}`, and `x` as a dictionary key |
| `__len__` | `len(x)`, and `if x:` when `__bool__` is absent |
| `__bool__` | `if x:` |
| `__lt__` | `x < y`, `sorted(items)`, `min` and `max` |

The family is much larger than this, covering arithmetic, indexing, iteration and the `with`
statement. The **Context Managers and Iterators** notebook takes the last two.

### `__eq__`: comparing by contents

Compare the attributes that decide whether two objects count as the same. Packing them into tuples
compares them in one line and stops at the first difference.

`isinstance` guards the case where the other side is not a station at all, and returning
`NotImplemented` is the correct answer: it tells Python that this class cannot answer, so it should
try the other object's `__eq__` before deciding they differ.


In [6]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r}, {self.unit!r})"

    def __eq__(self, other):
        if not isinstance(other, Station):
            return NotImplemented
        return ((self.name, self.readings, self.unit)
                == (other.name, other.readings, other.unit))


first = Station("Tromso", [-4.1], "C")
second = Station("Tromso", [-4.1], "C")
other = Station("Bodo", [-4.1], "C")

print("first == second:", first == second)
print("first is second:", first is second)
print("first == other: ", first == other)
print("first == 'Tromso':", first == "Tromso")


first == second: True
first is second: False
first == other:  False
first == 'Tromso': False


Equal contents, separate objects. `is` still answers the identity question, and now the two
questions have two different answers, which is the point.

`in` uses `==`, so it starts working too.


In [7]:
print("first in [second, other]:", first in [second, other])
print("index of an equal station:", [second, other].index(first))


first in [second, other]: True
index of an equal station: 0


### The rule that comes with `__eq__`

Defining `__eq__` and stopping there makes the object unusable in a set or as a dictionary key.


In [8]:
{first}


TypeError: cannot use 'Station' as a set element (unhashable type: 'Station')

Python did this on purpose. A set finds items by hash, and two objects that are equal must have the
same hash or the set will hold both and fail to find either. Since Python cannot know which
attributes your `__eq__` used, it removes the default hash rather than let it disagree with your
equality.

The fix is to hash the same values `__eq__` compares. `readings` is a list and lists cannot be
hashed, so it becomes a tuple.


In [9]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r}, {self.unit!r})"

    def __eq__(self, other):
        if not isinstance(other, Station):
            return NotImplemented
        return ((self.name, self.readings, self.unit)
                == (other.name, other.readings, other.unit))

    def __hash__(self):
        return hash((self.name, tuple(self.readings), self.unit))


first = Station("Tromso", [-4.1], "C")
second = Station("Tromso", [-4.1], "C")
other = Station("Bodo", [-4.1], "C")

print("set of three, two identical:", len({first, second, other}))
print("as a dictionary key:        ", {first: "north"}[second])


set of three, two identical: 2
as a dictionary key:         north


Three objects went into the set and two came out, because `first` and `second` are equal and now
hash alike.

One caution follows from that last line. `first` was used as the key and `second` found it, which
is what you asked for. It also means changing `first.name` after it is in the set leaves it filed
under its old hash, where nothing will find it. Objects used as keys should hold values that do not
change.

### `__len__`, and what it does to `if`

`__len__` makes `len(x)` work. It also decides truthiness, because an object with no `__bool__`
falls back to asking whether its length is zero.


In [10]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r}, {self.unit!r})"

    def __len__(self):
        return len(self.readings)


north = Station("Tromso", [-4.1, -2.6, -3.8])
empty = Station("Nowhere", [])

print("len(north):", len(north), " bool:", bool(north))
print("len(empty):", len(empty), " bool:", bool(empty))

if not empty:
    print("a station with no readings is falsy, without writing __bool__")


len(north): 3  bool: True
len(empty): 0  bool: False
a station with no readings is falsy, without writing __bool__


This is convenient and it is also a decision. Before writing `__len__`, be sure that "empty" is a
sensible thing for your object to be, because you have just made every `if station:` in the program
mean "if it has readings". When that is wrong, write `__bool__` and return what you actually mean.

### `__lt__`, so your objects sort

`sorted` needs to know which of two objects comes first. The **Why Classes** notebook did that with
`key=lambda s: min(s.in_celsius())` at every call site. `__lt__` puts the answer on the class.


In [11]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station({self.name!r}, {self.readings!r}, {self.unit!r})"

    def __lt__(self, other):
        if not isinstance(other, Station):
            return NotImplemented
        return min(self.readings) < min(other.readings)


group = [Station("Galway", [11.5]), Station("Tromso", [-4.1]), Station("Brno", [9.4])]

print("sorted:", sorted(group))
print("min:   ", min(group))
print("max:   ", max(group))


sorted: [Station('Tromso', [-4.1], 'C'), Station('Brno', [9.4], 'C'), Station('Galway', [11.5], 'C')]
min:    Station('Tromso', [-4.1], 'C')
max:    Station('Galway', [11.5], 'C')


`sorted`, `min` and `max` all work from that one method, because each of them only ever needs to ask
which of two items is smaller.

Define `__lt__` only when there is one obvious ordering. Stations could sort by name, by coldest
reading or by how many readings they hold, and picking one silently is worse than making every
caller pass a `key`.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/04-dunder-methods-solutions.ipynb).

**1.** Write a `Book` class storing `title`, `author` and `pages`, with a `__repr__` that reads like
the call that would rebuild it. Print one book and a list of two books.


In [12]:
# your code here


**2.** Add a `__str__` giving a friendly one-line description. Print the book, print `repr` of it,
and print a list containing it, so all three forms are visible at once.


In [13]:
# your code here


**3.** Add `__eq__` so two books with the same title and author are equal even if their page counts
differ. Show two equal books, two unequal books, and that comparing a book to a string is `False`
rather than an error.


In [14]:
# your code here


**4.** Put two equal books in a set. Read the error, then add `__hash__` so it works, and show that
the set holds one book rather than two.


In [15]:
# your code here


**5.** Add `__len__` returning the page count, then show `len(book)` and what `bool(book)` gives for
a book with `0` pages. Say in a comment whether `__len__` was a good choice here.


In [16]:
# your code here


**6.** Add `__lt__` so books sort by page count, then print a sorted list of three books and the
shortest one using `min`.


In [17]:
# your code here


## Common errors

### TypeError: unhashable, after defining `__eq__`

This is the error the worked examples met, and it is worth reading closely because the message
names the cause rather than the fix.


In [18]:
class OnlyEq:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return isinstance(other, OnlyEq) and self.name == other.name


{OnlyEq("Tromso")}


TypeError: cannot use 'OnlyEq' as a set element (unhashable type: 'OnlyEq')

`unhashable type` means `__hash__` is `None`. Defining `__eq__` sets it to `None` unless you also
define `__hash__`, so a class that has one and not the other cannot go in a set or be a dictionary
key.

If your object should not be a key, this is the correct end state and there is nothing to fix. If
it should, write `__hash__` over the same values `__eq__` compares.

### TypeError: `__repr__` returned something that is not a string

`__repr__` has to return a string. Returning the attribute directly is easy to do when the
attribute happens to be a number.


In [19]:
class NotAString:
    def __init__(self, pages):
        self.pages = pages

    def __repr__(self):
        return self.pages


repr(NotAString(412))


TypeError: __repr__ returned non-string (type int)

`__repr__ returned non-string (type int)` is exact. Wrap it: `return f"NotAString({self.pages!r})"`,
or at minimum `return str(self.pages)`.

### RecursionError: a `__repr__` that describes itself

Using `self` inside its own `__repr__` calls `__repr__` again, which calls it again.


In [20]:
class Loops:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Loops({self!r})"


try:
    repr(Loops("Tromso"))
except RecursionError:
    print("RecursionError: __repr__ called itself until the stack ran out")


RecursionError: __repr__ called itself until the stack ran out


This one is caught rather than left to raise, because the real message reports how much stack was
used, and that number differs between machines.

The fix is to format the **attributes**, not the object: `f"Loops({self.name!r})"`. The same
mistake happens with `f"{self}"` inside `__str__`.

### AttributeError: `__eq__` assumed the other side was the same class

`==` can be handed anything at all, including a string or `None`.


In [21]:
class Assumes:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        return self.name == other.name


Assumes("Tromso") == "Tromso"


AttributeError: 'str' object has no attribute 'name'

A string has no `.name`, so reaching for it raises rather than answering the question. Any code
doing `station in some_list` can trigger this, because `in` compares against every item.

Guard with `isinstance` and return `NotImplemented`, which is not the same as returning `False`.
`NotImplemented` tells Python you cannot answer, so it asks the other object, and only then decides
the two are unequal.


In [22]:
class Guarded:
    def __init__(self, name):
        self.name = name

    def __eq__(self, other):
        if not isinstance(other, Guarded):
            return NotImplemented
        return self.name == other.name


print("against a string:", Guarded("Tromso") == "Tromso")
print("against None:    ", Guarded("Tromso") == None)
print("against itself:  ", Guarded("Tromso") == Guarded("Tromso"))


against a string: False
against None:     False
against itself:   True


### The quiet one: `__str__` without `__repr__`

`str` falls back to `__repr__`, and the reverse is not true. A class with only `__str__` prints
nicely and then reverts the moment it is inside something.


In [23]:
class StrOnly:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"station {self.name}"


one = StrOnly("Tromso")

print("print(one):", one)
print("in a list: ", masked([one]))
print("in a dict: ", masked({"north": one}))


print(one): station Tromso
in a list:  [<__main__.StrOnly object at 0x...>]
in a dict:  {'north': <__main__.StrOnly object at 0x...>}


The first line looks correct, which is what makes this quiet. Nothing raises, and the class appears
to work until an object turns up in a list, a dictionary, a traceback or a debugger.

Write `__repr__` first. Add `__str__` only when the friendly form genuinely differs from the
debugging form, and accept that a class with just `__repr__` is complete.


## Recap

- A dunder method is one Python calls for you when your object is printed, compared, measured or
  sorted.
- `__repr__` is the debugging form. Make it read like the call that would rebuild the object.
- `!r` inside an f-string formats a value with its repr, which is what puts the quotes back on.
- Containers print their contents with `repr`, never `str`, which is why `__repr__` comes first.
- `__str__` is the form for people. `str` falls back to `__repr__`, and never the other way round.
- `__eq__` compares contents. `is` still compares identity, and now the two questions differ.
- `in` and `.index` use `__eq__`, so they start working once it exists.
- Defining `__eq__` removes the default hash, so a set or dictionary key needs `__hash__` too.
- Hash the same values `__eq__` compares, converting any list to a tuple.
- An object used as a dictionary key should hold values that do not change.
- `__len__` also decides truthiness, so writing it makes `if station:` mean "has readings".
- `__lt__` gives you `sorted`, `min` and `max`, and is worth writing only when one ordering is
  obvious.
- Return `NotImplemented` from `__eq__` for a type you cannot compare, rather than `False`.


## What is next

The **Context Managers and Iterators** notebook. Two more protocols, and the two you have used most
without writing either: `with open(...)` works because a file object defines `__enter__` and
`__exit__`, and `for row in reader` works because the reader defines `__iter__`. That notebook puts
both on a class of your own.


---

&#8592; **Previous:** [Methods](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/03-methods.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)  &nbsp;·&nbsp;  **Next:** [Context Managers and Iterators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/05-context-managers-and-iterators.ipynb) &#8594;
